<a href="https://colab.research.google.com/github/minji25-hue/NVIDIA/blob/20250404/yolov11_video.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# 1️⃣ 필수 라이브러리 설치
!apt-get install -y libgl1-mesa-glx
!pip install -U yt-dlp ultralytics opencv-python

import yt_dlp
import cv2
import time
import os
from ultralytics import YOLO
from google.colab.patches import cv2_imshow

# 2️⃣ YouTube 영상 다운로드 (yt-dlp 사용)
def download_youtube_video(url, output_path="traffic.mp4"):
    ydl_opts = {
        'format': 'bestvideo[ext=mp4]+bestaudio[ext=m4a]/best',
        'outtmpl': output_path,  # 저장할 파일 경로
    }
    try:
        with yt_dlp.YoutubeDL(ydl_opts) as ydl:
            ydl.download([url])
        print(f"✅ YouTube 영상 다운로드 완료: {output_path}")
    except Exception as e:
        print(f"❌ YouTube 다운로드 오류: {e}")

# 🎯 🔄 **새로운 YouTube 영상 URL 적용**
video_url = 'https://www.youtube.com/watch?v=QedNLyleW1w'  # 새로운 영상 URL
download_youtube_video(video_url, "/content/traffic.mp4")

# 3️⃣ YOLOv11x 모델 로드
model = YOLO('yolo11x.pt')  # 🔄 변경된 모델명

# 4️⃣ 차량 감지 및 카운팅
cap = cv2.VideoCapture("/content/traffic.mp4")

if not cap.isOpened():
    print("❌ 비디오 파일을 열 수 없습니다. 다운로드가 실패했을 가능성이 있습니다.")
else:
    frame_width = int(cap.get(3))
    frame_height = int(cap.get(4))
    fps = cap.get(cv2.CAP_PROP_FPS)  # FPS 정보 가져오기
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))  # 총 프레임 수
    duration = total_frames / fps  # 전체 영상 길이(초)

    print(f"🎥 영상 정보: {duration:.2f}초, {fps:.2f} FPS, 총 {total_frames} 프레임")

    # ✅ MP4V 코덱 사용
    out = cv2.VideoWriter('/content/output.mp4', cv2.VideoWriter_fourcc(*'MP4V'), int(fps), (frame_width, frame_height))

    # VideoWriter 정상 작동 여부 확인
    if not out.isOpened():
        print("❌ VideoWriter가 정상적으로 열리지 않았습니다. 코덱을 확인하세요.")
    else:
        print("✅ VideoWriter 초기화 성공")

    car_ids = set()

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        # 🚗 차량 감지 및 추적
        results = model.track(frame, persist=True, tracker="bytetrack.yaml", classes=[2,3,5,7], conf=0.5)

        if results and results[0].boxes.id is not None:
            for box_id in results[0].boxes.id.cpu().numpy().astype(int):
                car_ids.add(box_id)

        # 감지된 프레임 저장
        annotated_frame = results[0].plot() if results else frame
        out.write(annotated_frame)

    cap.release()
    out.release()

    # 5️⃣ 결과 출력
    print(f"🔹 전체 영상에서 통과한 차량 수: {len(car_ids)}대")
    print("\n📌 결과 영상 확인 방법:")
    print("1. 좌측 폴더 아이콘 📁 클릭")
    print("2. `/content/output.mp4` 파일 우클릭 → [다운로드] 또는 [미리보기]")

    # 6️⃣ output.mp4 파일 존재 여부 확인 후 다운로드
    output_path = "/content/output.mp4"
    if os.path.exists(output_path):
        print("✅ output.mp4 파일이 정상적으로 저장되었습니다.")
        from google.colab import files
        files.download(output_path)
    else:
        print("❌ output.mp4 파일이 생성되지 않았습니다. OpenCV 설정을 확인하세요.")

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  libgl1-mesa-glx
0 upgraded, 1 newly installed, 0 to remove and 30 not upgraded.
Need to get 5,584 B of archives.
After this operation, 74.8 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy-updates/universe amd64 libgl1-mesa-glx amd64 23.0.4-0ubuntu1~22.04.1 [5,584 B]
Fetched 5,584 B in 0s (15.5 kB/s)
Selecting previously unselected package libgl1-mesa-glx:amd64.
(Reading database ... 126210 files and directories currently installed.)
Preparing to unpack .../libgl1-mesa-glx_23.0.4-0ubuntu1~22.04.1_amd64.deb ...
Unpacking libgl1-mesa-glx:amd64 (23.0.4-0ubuntu1~22.04.1) ...
Setting up libgl1-mesa-glx:amd64 (23.0.4-0ubuntu1~22.04.1) ...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 172.2/172.2 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 41.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━

100%|██████████| 109M/109M [00:00<00:00, 118MB/s]


🎥 영상 정보: 39.01초, 29.97 FPS, 총 1169 프레임
✅ VideoWriter 초기화 성공
requirements: Ultralytics requirement ['lap>=0.5.12'] not found, attempting AutoUpdate...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 35.8 MB/s eta 0:00:00

requirements: AutoUpdate success ✅ 2.6s, installed 1 package: ['lap>=0.5.12']
requirements: ⚠️ Restart runtime or rerun command for updates to take effect


0: 384x640 8 cars, 1 bus, 74.5ms
Speed: 17.0ms preprocess, 74.5ms inference, 472.2ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 8 cars, 1 bus, 55.8ms
Speed: 3.4ms preprocess, 55.8ms inference, 1.6ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 8 cars, 1 bus, 55.7ms
Speed: 3.2ms preprocess, 55.7ms inference, 1.5ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 8 cars, 1 bus, 55.7ms
Speed: 2.3ms preprocess, 55.7ms inference, 1.6ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 8 cars, 1 bus, 43.2ms
Speed: 2.8ms preprocess, 43.2ms inference, 1.6ms postpr

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>